## 1. Config

In [1]:
from pathlib import Path

CONFIG = {
    "MODEL_PATH": "runs/detect/dcp1_exp1-3/weights/best.pt",   # <-- your trained model
    "YOLO_DATASET_DIR": "data/yolo_dataset",                  # <-- root with images/ and labels/ per split
    "SPLITS": ["train", "val", "test"],
    "SUSPICIOUS_DIR": "data/suspicious_review",

    # how wide a net to cast when running inference (low, so we see borderline detections too)
    "INFERENCE_CONF": 0.05,
    # a GT box counts as 'confirmed' only if a same-class prediction overlaps it by at least
    # IOU_MATCH_THRESHOLD AND has confidence at least CONFIRM_CONF_THRESHOLD
    "IOU_MATCH_THRESHOLD": 0.3,
    "CONFIRM_CONF_THRESHOLD": 0.25,

    # flag an image if it has at least this many unconfirmed GT boxes
    "MIN_UNCONFIRMED_TO_FLAG": 1,

    "DEVICE": 0,
}

Path(CONFIG["SUSPICIOUS_DIR"]).mkdir(parents=True, exist_ok=True)

## 2. Load the trained model

In [2]:
from ultralytics import YOLO

model = YOLO(CONFIG["MODEL_PATH"])
names = model.names  # {class_id: class_name}, should match your data.yaml
print(names)

{0: 'Bike-car-person', 1: 'Bike', 2: 'Car', 3: 'Person'}


## 3. Helper functions: load GT labels, IoU, matching

In [3]:
import cv2
import numpy as np

def load_yolo_labels(label_path, img_w, img_h):
    """Returns list of (class_id, x1, y1, x2, y2) in absolute pixel coords."""
    boxes = []
    if not Path(label_path).exists():
        return boxes
    for line in Path(label_path).read_text().strip().splitlines():
        if not line.strip():
            continue
        cls, cx, cy, w, h = line.split()
        cls = int(cls)
        cx, cy, w, h = float(cx) * img_w, float(cy) * img_h, float(w) * img_w, float(h) * img_h
        x1, y1 = cx - w / 2, cy - h / 2
        x2, y2 = cx + w / 2, cy + h / 2
        boxes.append((cls, x1, y1, x2, y2))
    return boxes

def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def confirm_gt_boxes(gt_boxes, pred_boxes, pred_confs, pred_classes,
                      iou_thresh=CONFIG["IOU_MATCH_THRESHOLD"], conf_thresh=CONFIG["CONFIRM_CONF_THRESHOLD"]):
    """For each GT box, find the best same-class prediction and decide if it's confirmed.
    Returns a list of dicts: {gt_idx, class_id, best_iou, best_conf, confirmed}."""
    results = []
    for i, (cls, x1, y1, x2, y2) in enumerate(gt_boxes):
        best_iou, best_conf = 0.0, 0.0
        for (px1, py1, px2, py2), pconf, pcls in zip(pred_boxes, pred_confs, pred_classes):
            if pcls != cls:
                continue
            cur_iou = iou((x1, y1, x2, y2), (px1, py1, px2, py2))
            if cur_iou > best_iou:
                best_iou, best_conf = cur_iou, pconf
        confirmed = best_iou >= iou_thresh and best_conf >= conf_thresh
        results.append({"gt_idx": i, "class_id": cls, "best_iou": best_iou,
                        "best_conf": best_conf, "confirmed": confirmed})
    return results

## 4. Run inference across every split and score each image
For every image: run the model at a low confidence threshold, load its GT labels, and check which GT boxes go unconfirmed. Nothing is written yet — this just builds the report.

In [4]:
import pandas as pd

rows = []
gt_confirmations = {}  # (split, filename) -> list of confirm_gt_boxes() results
pred_cache = {}        # (split, filename) -> (pred_boxes, pred_confs, pred_classes)

for split in CONFIG["SPLITS"]:
    img_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "images" / split
    label_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "labels" / split
    image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")])

    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        gt_boxes = load_yolo_labels(label_dir / f"{img_path.stem}.txt", w, h)

        result = model.predict(img, conf=CONFIG["INFERENCE_CONF"], device=CONFIG["DEVICE"], verbose=False)[0]
        pred_boxes = result.boxes.xyxy.cpu().numpy().tolist() if len(result.boxes) else []
        pred_confs = result.boxes.conf.cpu().numpy().tolist() if len(result.boxes) else []
        pred_classes = result.boxes.cls.cpu().numpy().astype(int).tolist() if len(result.boxes) else []

        confirmations = confirm_gt_boxes(gt_boxes, pred_boxes, pred_confs, pred_classes)
        n_unconfirmed = sum(1 for c in confirmations if not c["confirmed"])

        key = (split, img_path.name)
        gt_confirmations[key] = confirmations
        pred_cache[key] = (pred_boxes, pred_confs, pred_classes)

        rows.append({
            "split": split,
            "filename": img_path.name,
            "num_gt": len(gt_boxes),
            "num_unconfirmed": n_unconfirmed,
            "suspicion_score": n_unconfirmed / len(gt_boxes) if gt_boxes else 0.0,
            "flagged": n_unconfirmed >= CONFIG["MIN_UNCONFIRMED_TO_FLAG"],
        })

report_df = pd.DataFrame(rows).sort_values("suspicion_score", ascending=False)
report_df.to_csv(Path(CONFIG["SUSPICIOUS_DIR"]) / "suspicious_report.csv", index=False)
print(report_df["flagged"].sum(), "images flagged out of", len(report_df))
report_df.head(20)

5380 images flagged out of 10922


,split,filename,num_gt,num_unconfirmed,suspicion_score,flagged
2,train,A-D-5_jpg.rf.be64f9a6f22cfa384e9472e9f44f79e5.jpg,4,4,1.0,True
25,train,A-F-36_jpg.rf.6f480ec63271f817db8bb14e32ae86af...,4,4,1.0,True
10,train,A-F-12_jpg.rf.fa2fd052854bf08b6c78f49fdc20c3b1...,2,2,1.0,True
24,train,A-F-34_jpg.rf.efa403f8c54ef9a0ce2a5abc2c0be22e...,4,4,1.0,True
0,train,A-D-1_jpg.rf.e563fc3736cc268601df1977200502cb.jpg,1,1,1.0,True
3,train,A-D-5_jpg.rf.d55c9811122697ec6efcf8bbb8774d95.jpg,4,4,1.0,True
17,train,A-F-23_jpg.rf.71f692b2ec36e47c8076a595fde3a6ca...,1,1,1.0,True
18,train,A-F-25_jpg.rf.a2bfcaa407b4c93db7e2891e6433079d...,1,1,1.0,True
19,train,A-F-29_jpg.rf.b47dd3fd1dba9d799060430e29735da3...,4,4,1.0,True
21,train,A-F-31_jpg.rf.292313da1b05f176d350414bc126ccd2...,4,4,1.0,True


## 5. Save side-by-side visualizations for flagged images
Red = ground-truth boxes (thicker/labeled 'UNCONFIRMED' if the model never agreed with them, thin otherwise). Green = model predictions at or above `CONFIRM_CONF_THRESHOLD`. Saved into `SUSPICIOUS_DIR/<split>/`, one image per flagged file, so you can flip through them like the examples you sent.

In [5]:
for split in CONFIG["SPLITS"]:
    out_dir = Path(CONFIG["SUSPICIOUS_DIR"]) / split
    out_dir.mkdir(parents=True, exist_ok=True)

    img_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "images" / split
    label_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "labels" / split

    flagged_names = report_df[(report_df["split"] == split) & (report_df["flagged"])]["filename"]

    for filename in flagged_names:
        key = (split, filename)
        img_path = img_dir / filename
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gt_boxes = load_yolo_labels(label_dir / f"{Path(filename).stem}.txt", w, h)
        confirmations = gt_confirmations[key]
        pred_boxes, pred_confs, pred_classes = pred_cache[key]

        vis = img.copy()
        # predictions in green
        for (px1, py1, px2, py2), pconf, pcls in zip(pred_boxes, pred_confs, pred_classes):
            if pconf < CONFIG["CONFIRM_CONF_THRESHOLD"]:
                continue
            cv2.rectangle(vis, (int(px1), int(py1)), (int(px2), int(py2)), (0, 200, 0), 2)
            cv2.putText(vis, f"{names[pcls]} {pconf:.2f}", (int(px1), max(int(py1) - 6, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 2)
        # ground truth in red; unconfirmed ones thicker + labeled
        for (cls, x1, y1, x2, y2), conf_info in zip(gt_boxes, confirmations):
            thickness = 3 if not conf_info["confirmed"] else 1
            label = f"{names[cls]} GT" + (" - UNCONFIRMED" if not conf_info["confirmed"] else "")
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), thickness)
            cv2.putText(vis, label, (int(x1), min(int(y2) + 16, h - 1)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        cv2.imwrite(str(out_dir / f"{Path(filename).stem}.jpg"), vis)

    print(f"{split}: wrote {len(flagged_names)} review images to {out_dir}")

train: wrote 3633 review images to data\suspicious_review\train
val: wrote 994 review images to data\suspicious_review\val
test: wrote 753 review images to data\suspicious_review\test


## 6. Manual review step (do this yourself, outside the notebook)
Open `SUSPICIOUS_DIR/<split>/` for each split. Each image shows red GT boxes (thick + 'UNCONFIRMED' where the model never agreed) and green model predictions.

- **If the red box is genuinely bogus** (your `Car`-on-empty-road case) — the label is bad — **leave the review image where it is.**
- **If the red box actually is a real object the model just missed** (a legitimately hard example, e.g. small/occluded) — the label is fine — **delete that review image from the `SUSPICIOUS_DIR` folder.**

Whatever review images remain after you're done is exactly what the next cell removes from the dataset — nothing is deleted automatically before this step.

## 7. Remove the confirmed-bad images from the dataset
For safety this **moves** the matching original image + label into `YOLO_DATASET_DIR/_removed/<split>/` rather than permanently deleting — easy to restore if you change your mind about one later.

In [6]:
import shutil

removed_counts = {}
for split in CONFIG["SPLITS"]:
    review_dir = Path(CONFIG["SUSPICIOUS_DIR"]) / split
    remaining_stems = {p.stem for p in review_dir.glob("*.jpg")} if review_dir.exists() else set()

    img_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "images" / split
    label_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "labels" / split
    removed_img_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "_removed" / split / "images"
    removed_label_dir = Path(CONFIG["YOLO_DATASET_DIR"]) / "_removed" / split / "labels"
    removed_img_dir.mkdir(parents=True, exist_ok=True)
    removed_label_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    for stem in remaining_stems:
        matches = list(img_dir.glob(f"{stem}.*"))
        for img_path in matches:
            shutil.move(str(img_path), str(removed_img_dir / img_path.name))
        label_path = label_dir / f"{stem}.txt"
        if label_path.exists():
            shutil.move(str(label_path), str(removed_label_dir / label_path.name))
        if matches:
            count += 1

    removed_counts[split] = count
    print(f"{split}: removed {count} images (backed up under _removed/{split}/)")

print("Total removed:", sum(removed_counts.values()))
print("Re-run section 8/9 of the training notebook (label conversion already baked in, just re-point data.yaml counts) if you want updated split sizes reflected before retraining.")

train: removed 0 images (backed up under _removed/train/)
val: removed 985 images (backed up under _removed/val/)
test: removed 742 images (backed up under _removed/test/)
Total removed: 1727
Re-run section 8/9 of the training notebook (label conversion already baked in, just re-point data.yaml counts) if you want updated split sizes reflected before retraining.
